In [3]:
import subprocess
from pathlib import Path

def extract(input_dir, output_csv):
    pcaps = sorted(p for p in Path(input_dir).glob("*.pcap") if p.name != "merged.pcap")
    merged_pcap = Path(input_dir) / "merged.pcap"

    # Step 1: merge all pcaps into one
    merge_cmd = ["mergecap", "-w", str(merged_pcap)] + [str(p) for p in pcaps]
    print(f"Merging {len(pcaps)} pcap(s)...")
    subprocess.run(merge_cmd, check=True)

    # Step 2: run tshark on the merged file
    cmd = [
        "tshark", "-r", str(merged_pcap),
        "-T", "fields",
        # Frame
        "-e", "frame.time_epoch",
        "-e", "frame.len",
        # IP
        "-e", "ip.src",
        "-e", "ip.dst",
        # TCP
        "-e", "tcp.srcport",
        "-e", "tcp.dstport",
        "-e", "tcp.seq",
        "-e", "tcp.ack",
        "-e", "tcp.len",
        "-e", "tcp.analysis.retransmission",
        # MBAP header
        "-e", "mbtcp.trans_id",
        "-e", "mbtcp.unit_id",
        "-e", "mbtcp.len",
        "-e", "mbtcp.prot_id",
        # Modbus function & addressing
        "-e", "modbus.func_code",
        "-e", "modbus.exception_code",
        "-e", "modbus.reference_num",
        "-e", "modbus.regnum16",
        "-e", "modbus.bitnum",
        # Modbus counts
        "-e", "modbus.word_cnt",
        "-e", "modbus.byte_cnt",
        "-e", "modbus.bit_cnt",
        # Modbus values
        "-e", "modbus.regval_uint16",
        "-e", "modbus.data",
        # Modbus request/response linking
        "-e", "modbus.response_time",
        "-e", "modbus.request_frame",
        # Output format
        "-E", "header=y",
        "-E", "separator=,",
        "-E", "quote=d",
        "-E", "occurrence=a",
        "-E", "aggregator=|",
    ]

    print("Extracting fields...")
    with open(output_csv, "w") as f:
        subprocess.run(cmd, stdout=f, check=True)

    print("Saved:", output_csv)

In [4]:
extract("../data/benign/network-wide-pcap-capture/network-wide","../train/benign_nw_analysis.csv")

Merging 19 pcap(s)...
Extracting fields...
Saved: ../train/benign_nw_analysis.csv


In [5]:
extract("../data/attack/compromised-scada/substation-wide-capture", "../train/cscada_attack_ssw_analysis.csv")

Merging 18 pcap(s)...
Extracting fields...
Saved: ../train/cscada_attack_ssw_analysis.csv


In [6]:
extract("../data/attack/external/network-wide", "../train/ext_attack_nw_analysis.csv")

Merging 2 pcap(s)...
Extracting fields...
Saved: ../train/ext_attack_nw_analysis.csv


In [7]:
import os
os.system('notify-send "Python Script" "Execution complete!"')

0